# Taller de YOLO - Ejercicio 1: Detección personalizada de extremo a extremo
Mismo flujo que el cuaderno `YOLO - Ej1.ipynb`: dataset → `data.yaml` → entrenamiento → métricas → mAP con `supervision` → visualización.

*Dataset:* **medical-pills** (Ultralytics, formato YOLO, 1 clase `pill`, 92 imágenes train / 23 val). Se usa en lugar de `circuit-voltages` de Roboflow porque su descarga no requiere cuenta ni API key de Roboflow; el procedimiento es idéntico.

In [1]:
import os, torch, ultralytics
HOME = os.getcwd()
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| dispositivo:", "cuda" if torch.cuda.is_available() else "cpu")

ultralytics 8.4.156 | torch 2.14.0+cu130 | dispositivo: cpu


## 1. Dataset

In [2]:
dataset_location = f"{HOME}/medical-pills"
!ls {dataset_location} {dataset_location}/images
!ls {dataset_location}/images/train | head -3; ls {dataset_location}/labels/train | wc -l
!head -3 {dataset_location}/labels/train/Frame_0.txt

/home/aramber/ta04/ej1/medical-pills:
LICENSE.txt  images  labels  medical-pills.yaml

/home/aramber/ta04/ej1/medical-pills/images:
train  val


Frame_0.jpg
Frame_100.jpg
Frame_104.jpg
92


0 0.330729 0.816204 0.103125 0.121296
0 0.347656 0.658333 0.094271 0.138889
0 0.498958 0.626852 0.105208 0.144444


## 2. Normalización del `data.yaml`

In [3]:
import yaml, shutil
shutil.copy(f'{dataset_location}/medical-pills.yaml', f'{dataset_location}/data.yaml')
yaml_path = f'{dataset_location}/data.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data['path'] = dataset_location
data['train'] = 'images/train'
data['val'] = 'images/val'
data.pop('download', None)
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
!cat {yaml_path}

path: /home/aramber/ta04/ej1/medical-pills
train: images/train
val: images/val
names:
  0: pill


## 3. Entrenamiento (10 épocas)

In [4]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(data=yaml_path, epochs=10, imgsz=640, batch=8, workers=2, device='cpu', name='train_e10', exist_ok=True, plots=True, verbose=False)

Ultralytics 8.4.156 🚀 Python-3.11.15 torch-2.14.0+cu130 CPU (Intel Xeon Processor @ 2.10GHz)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/aramber/ta04/ej1/medical-pills/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_e10, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspec

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 1/2 5.2s/it 1.6s<5.2s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4s/it 2.8s

                   all         23        399      0.976      0.985      0.992      0.752


Speed: 1.8ms preprocess, 87.0ms inference, 0.0ms loss, 13.3ms postprocess per image


Results saved to /home/aramber/ta04/ej1/runs/detect/train_e10


## 4. Métricas del entrenamiento

In [5]:
import glob
run_dir = f'{HOME}/runs/detect/train_e10'
print(sorted(os.listdir(run_dir)))
print(open(f'{run_dir}/results.csv').read())

['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'args.yaml', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'labels.jpg', 'results.csv', 'results.png', 'train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg', 'val_batch1_labels.jpg', 'val_batch1_pred.jpg', 'weights']
epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
1,42.7236,1.38039,2.86798,1.15225,0.05739,0.99248,0.70901,0.35697,0.97808,2.8432,0.88829,0.000611111,0.000611111,0.000611111
2,81.2122,1.16322,1.80519,0.98624,0.05725,0.98997,0.82993,0.412,1.21869,2.75286,0.95613,0.00115128,0.00115128,0.00115128
3,119.052,1.2274,1.27562,1.03317,0.05754,0.99499,0.75333,0.39758,1.16079,2.58837,0.94653,0.00155944,0.00155944,0.00155944
4,158.516,1.13861,0.9995,1.02935,0.10962,0.97744,0.9,0.60784,1.12322,2.37313,0.962

## 5. Cálculo de mAP con `supervision` (split de validación: el dataset no trae test)

In [6]:
import supervision as sv
from supervision.metrics import MeanAveragePrecision
best = os.path.join(run_dir, 'weights', 'best.pt')
model = YOLO(best)
ds = sv.DetectionDataset.from_yolo(images_directory_path=f'{dataset_location}/images/val', annotations_directory_path=f'{dataset_location}/labels/val', data_yaml_path=yaml_path)
print('Clases:', ds.classes)
print('Imágenes en validación:', len(ds))

Clases: ['pill']
Imágenes en validación: 23


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
predictions, targets = [], []
for _, image, target in ds:
    r = model(image, verbose=False)[0]
    predictions.append(sv.Detections.from_ultralytics(r)); targets.append(target)
mapr = MeanAveragePrecision().update(predictions, targets).compute()
print(f'mAP@50:95: {mapr.map50_95:.3f}')
print(f'mAP@50:    {mapr.map50:.3f}')
print(f'mAP@75:    {mapr.map75:.3f}')

mAP@50:95: 0.728
mAP@50:    0.948
mAP@75:    0.908


## 6. Visualización de una predicción

In [8]:
import random, cv2
random.seed(3)
i = random.randint(0, len(ds) - 1)
image_path, image, target = ds[i]
r = model(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(r).with_nms()
ann = sv.BoxAnnotator().annotate(scene=image.copy(), detections=detections)
ann = sv.LabelAnnotator().annotate(scene=ann, detections=detections, labels=[f'pill {c:.2f}' for c in detections.confidence])
cv2.imwrite('prediccion.jpg', ann)
print(os.path.basename(image_path), '-> detecciones:', len(detections), '| reales:', len(target))

Frame_64.jpg -> detecciones: 16 | reales: 16


## 7. Pregunta 5: ¿1 época? ¿5 épocas?

In [9]:
res = {}
for ep in [1, 5]:
    m = YOLO('yolo11n.pt')
    m.train(data=yaml_path, epochs=ep, imgsz=640, batch=8, workers=2, device='cpu', name=f'train_e{ep}', exist_ok=True, plots=False, verbose=False)
    met = YOLO(f'{HOME}/runs/detect/train_e{ep}/weights/best.pt').val(data=yaml_path, device='cpu', verbose=False, plots=False)
    res[ep] = (met.box.mp, met.box.mr, met.box.map50, met.box.map)
met = YOLO(best).val(data=yaml_path, device='cpu', verbose=False, plots=False)
res[10] = (met.box.mp, met.box.mr, met.box.map50, met.box.map)
import pandas as pd
print(pd.DataFrame(res, index=['precision','recall','mAP50','mAP50-95']).T.round(3))

Ultralytics 8.4.156 🚀 Python-3.11.15 torch-2.14.0+cu130 CPU (Intel Xeon Processor @ 2.10GHz)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/aramber/ta04/ej1/medical-pills/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_e1, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 1/2 3.5s/it 1.0s<3.5s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.5s

                   all         23        399      0.976      0.985      0.992      0.752


Speed: 0.9ms preprocess, 48.1ms inference, 0.0ms loss, 10.1ms postprocess per image


    precision  recall  mAP50  mAP50-95
1       0.000   0.000  0.000     0.000
5       0.959   0.877  0.918     0.654
10      0.976   0.985  0.992     0.752
